# 02 — Tiền xử lý WESAD: Common-rate (64 Hz)

Notebook này được tách riêng khỏi `01_eda.ipynb`. Nó đọc trực tiếp 15 file `.pkl` gốc, không phụ thuộc vào biến trong notebook EDA.

**Mục tiêu:** Resampling 10 loại tín hiệu / 14 kênh về cùng tần số thử nghiệm **64 Hz**, lưu một ma trận `(N, 14)` cho mỗi subject, đồng thời bảo toàn các đoạn nhãn gốc (700 Hz) và thông tin kiểm soát chất lượng.

**Lưu ý nghiên cứu:** 64 Hz chỉ là cấu hình ban đầu; tín hiệu ngực 700 Hz bị giảm băng thông, đặc biệt có thể ảnh hưởng tới ECG và EMG. Các phép resampling không chứng minh tính phù hợp sinh lý học hay độ chính xác mô hình. **Chưa chia cửa sổ, chuẩn hóa hay tạo tập train/test** ở notebook này.

## 1. Cấu hình dữ liệu và đầu ra

- **Input:** `data/raw/WESAD/SX/SX.pkl` (15 subject).
- **Output:** `data/processed/common_rate_64hz/`.
- **Đầu ra:** `SX_X.npy`, `SX_quality_mask.npy`, `label_segments.csv`, `subjects_summary.csv`, `config.json`.
- `SX_X.npy`: 14 kênh được resampling về 64 Hz, lưu `float32`.
- `SX_quality_mask.npy`: mẫu 64 Hz có thể sử dụng theo quy tắc kiểm soát chất lượng ban đầu; `False` nghĩa là thuộc vùng đã đánh dấu.
- `label_segments.csv`: ranh giới nhãn theo **chỉ số mẫu gốc 700 Hz**, không ép nhãn về 64 Hz.
- Chưa áp dụng chuẩn hóa để tránh rò rỉ thông tin từ subject test ở giai đoạn LOSO.

In [1]:
from pathlib import Path
from math import gcd
import pickle
import json
import gc

import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from IPython.display import display

# Cho phép chạy notebook từ thư mục notebooks/ hoặc gốc dự án
PROJECT_ROOT = Path("..") if Path("../data/raw/WESAD").exists() else Path(".")
RAW_ROOT = PROJECT_ROOT / "data/raw/WESAD"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/common_rate_64hz"

SUBJECT_IDS = [
    "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9",
    "S10", "S11", "S13", "S14", "S15", "S16", "S17"
]

LABEL_FS = 700
TARGET_FS = 64
QUALITY_GUARD_SECONDS = 2.0  # Khoảng đệm thử nghiệm, chưa phải ngưỡng đã kiểm chứng

SIGNAL_CONFIG = {
    "chest": {"ACC": 700, "ECG": 700, "EMG": 700,
              "EDA": 700, "Temp": 700, "Resp": 700},
    "wrist": {"ACC": 32, "BVP": 64, "EDA": 4, "TEMP": 4},
}

CHANNEL_NAMES = [
    "chest_ACC_X", "chest_ACC_Y", "chest_ACC_Z",
    "chest_ECG", "chest_EMG", "chest_EDA", "chest_Temp", "chest_Resp",
    "wrist_ACC_X", "wrist_ACC_Y", "wrist_ACC_Z",
    "wrist_BVP", "wrist_EDA", "wrist_TEMP",
]

assert RAW_ROOT.exists(), f"Không tìm thấy thư mục: {RAW_ROOT.resolve()}"
assert len(CHANNEL_NAMES) == 14
for subject in SUBJECT_IDS:
    assert (RAW_ROOT / subject / f"{subject}.pkl").is_file(), f"Thiếu {subject}.pkl"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Nguồn:", RAW_ROOT.resolve())
print("Đầu ra:", OUTPUT_ROOT.resolve())
print("Subjects:", len(SUBJECT_IDS), "| Channels:", len(CHANNEL_NAMES))

Nguồn: D:\SGU26_DeepLearning_Group\Projects\DL13-WESAD\data\raw\WESAD
Đầu ra: D:\SGU26_DeepLearning_Group\Projects\DL13-WESAD\data\processed\common_rate_64hz
Subjects: 15 | Channels: 14


## 2. Resampling và lưu theo subject

Mỗi subject được đọc và xử lý lần lượt, tránh giữ cả 15 bản ghi trong RAM. `resample_poly` sử dụng bộ lọc trước khi lấy mẫu lại. Thay vì resampling riêng từng window, ta xử lý **toàn bộ chuỗi thời gian** rồi lưu `.npy` bằng memory map.

Nhãn được giữ ở dạng các **đoạn liên tiếp (label segments)** để việc gán nhãn sau này dựa trên khoảng thời gian thực. Vùng đáng nghi S3 được giữ trong dữ liệu đã resampling nhưng sẽ được đánh dấu trong `quality_mask`, mở rộng 2 giây mỗi bên theo một giả định bảo thủ cần đánh giá sau này.

**Quan trọng:** Chạy cell này sẽ tạo/ghi đè các file trong `common_rate_64hz/`. Dữ liệu gốc `.pkl` không bị thay đổi.

In [2]:
def convert_sampling_rate(values, original_fs, target_fs):
    """Lấy mẫu lại một mảng (samples, channels) bằng polyphase FIR."""
    if original_fs == target_fs:
        return values
    common_divisor = gcd(original_fs, target_fs)
    return resample_poly(
        values,
        up=target_fs // common_divisor,
        down=original_fs // common_divisor,
        axis=0,
    )

summary_rows = []
label_segment_rows = []

for subject_id in SUBJECT_IDS:
    input_file = RAW_ROOT / subject_id / f"{subject_id}.pkl"
    with open(input_file, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    labels = data["label"]
    duration_sec = len(labels) / LABEL_FS
    expected_rows = round(duration_sec * TARGET_FS)

    # Lưu trực tiếp lên đĩa; shape = (số mốc 64 Hz, 14 kênh)
    X = np.lib.format.open_memmap(
        OUTPUT_ROOT / f"{subject_id}_X.npy",
        mode="w+",
        dtype=np.float32,
        shape=(expected_rows, len(CHANNEL_NAMES)),
    )

    column_start = 0
    for device, signal_specs in SIGNAL_CONFIG.items():
        for signal_name, original_fs in signal_specs.items():
            original = data["signal"][device][signal_name]
            converted = convert_sampling_rate(original, original_fs, TARGET_FS)

            if converted.shape[0] != expected_rows:
                raise ValueError(
                    f"{subject_id} {device}/{signal_name}: "
                    f"expected {expected_rows}, got {converted.shape[0]}"
                )
            n_channels = converted.shape[1]
            X[:, column_start:column_start + n_channels] = converted
            column_start += n_channels
            del converted

    if column_start != len(CHANNEL_NAMES):
        raise ValueError(f"{subject_id}: expected 14 channels, got {column_start}")
    X.flush()

    # Ranh giới nhãn ở độ phân giải nguyên bản 700 Hz; end không bao gồm
    change_points = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    boundaries = np.r_[0, change_points, len(labels)]
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        label_segment_rows.append({
            "Subject": subject_id,
            "Label": int(labels[start]),
            "Start_sample_700": int(start),
            "End_sample_700": int(end),
            "Start_sec": start / LABEL_FS,
            "End_sec": end / LABEL_FS,
        })

    # Chỉ đánh dấu những mẫu thỏa tiêu chí bất thường đã khảo sát trong EDA
    chest_temp = data["signal"]["chest"]["Temp"][:, 0]
    chest_acc = data["signal"]["chest"]["ACC"]
    suspicious_temp = np.isclose(chest_temp, -273.15, atol=0.01, rtol=0)
    suspicious_acc = np.all(
        np.isclose(chest_acc, -6.6, atol=0.001, rtol=0), axis=1
    )
    bad_indices = np.flatnonzero(suspicious_temp | suspicious_acc)

    quality_mask = np.ones(expected_rows, dtype=bool)
    if len(bad_indices):
        discontinuities = np.flatnonzero(np.diff(bad_indices) > 1)
        starts = bad_indices[np.r_[0, discontinuities + 1]]
        ends = bad_indices[np.r_[discontinuities, len(bad_indices) - 1]] + 1
        for start, end in zip(starts, ends):
            t0 = max(0.0, start / LABEL_FS - QUALITY_GUARD_SECONDS)
            t1 = min(duration_sec, end / LABEL_FS + QUALITY_GUARD_SECONDS)
            i0 = max(0, int(np.floor(t0 * TARGET_FS)))
            i1 = min(expected_rows, int(np.ceil(t1 * TARGET_FS)))
            quality_mask[i0:i1] = False

    np.save(OUTPUT_ROOT / f"{subject_id}_quality_mask.npy", quality_mask)

    summary_rows.append({
        "Subject": subject_id,
        "Duration_sec": duration_sec,
        "Output_samples": expected_rows,
        "Channels": len(CHANNEL_NAMES),
        "Target_Hz": TARGET_FS,
        "Suspicious_samples_700": len(bad_indices),
        "Invalid_samples_64": int((~quality_mask).sum()),
    })

    print(f"[OK] {subject_id}: X={X.shape}, raw_suspicious={len(bad_indices)}")
    del X, data, labels, chest_temp, chest_acc
    del suspicious_temp, suspicious_acc, bad_indices, quality_mask
    gc.collect()

# Chỉ ghi metadata sau khi hoàn thành 15 subject
pd.DataFrame(summary_rows).to_csv(OUTPUT_ROOT / "subjects_summary.csv", index=False)
pd.DataFrame(label_segment_rows).to_csv(OUTPUT_ROOT / "label_segments.csv", index=False)

with open(OUTPUT_ROOT / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "strategy": "common_rate",
        "target_fs": TARGET_FS,
        "label_source_fs": LABEL_FS,
        "channels": CHANNEL_NAMES,
        "subjects": SUBJECT_IDS,
        "quality_guard_seconds": QUALITY_GUARD_SECONDS,
        "resampling_method": "scipy.signal.resample_poly",
        "normalized": False,
        "windowed": False,
    }, f, indent=2, ensure_ascii=False)

display(pd.DataFrame(summary_rows))
print("Đã lưu tại:", OUTPUT_ROOT.resolve())

[OK] S2: X=(389056, 14), raw_suspicious=0
[OK] S3: X=(415552, 14), raw_suspicious=83
[OK] S4: X=(411072, 14), raw_suspicious=0
[OK] S5: X=(400512, 14), raw_suspicious=0
[OK] S6: X=(452544, 14), raw_suspicious=0
[OK] S7: X=(335232, 14), raw_suspicious=0
[OK] S8: X=(349824, 14), raw_suspicious=0
[OK] S9: X=(334272, 14), raw_suspicious=0
[OK] S10: X=(351744, 14), raw_suspicious=0
[OK] S11: X=(334912, 14), raw_suspicious=0
[OK] S13: X=(354368, 14), raw_suspicious=0
[OK] S14: X=(355072, 14), raw_suspicious=0
[OK] S15: X=(336128, 14), raw_suspicious=0
[OK] S16: X=(360384, 14), raw_suspicious=0
[OK] S17: X=(378880, 14), raw_suspicious=0


,Subject,Duration_sec,Output_samples,Channels,Target_Hz,Suspicious_samples_700,Invalid_samples_64
0,S2,6079.0,389056,14,64,0,0
1,S3,6493.0,415552,14,64,83,265
2,S4,6423.0,411072,14,64,0,0
3,S5,6258.0,400512,14,64,0,0
4,S6,7071.0,452544,14,64,0,0
5,S7,5238.0,335232,14,64,0,0
6,S8,5466.0,349824,14,64,0,0
7,S9,5223.0,334272,14,64,0,0
8,S10,5496.0,351744,14,64,0,0
9,S11,5233.0,334912,14,64,0,0


Đã lưu tại: D:\SGU26_DeepLearning_Group\Projects\DL13-WESAD\data\processed\common_rate_64hz


## 3. Kiểm tra các file sau khi lưu

Yêu cầu đạt: 15 subject, mỗi ma trận có 14 cột, không có giá trị không hữu hạn, và độ dài `quality_mask` bằng số hàng của ma trận. Thao tác `mmap_mode="r"` giúp đọc dữ liệu lớn mà không tải cả ma trận vào RAM.

In [3]:
verification = []
for subject_id in SUBJECT_IDS:
    X = np.load(OUTPUT_ROOT / f"{subject_id}_X.npy", mmap_mode="r")
    quality_mask = np.load(
        OUTPUT_ROOT / f"{subject_id}_quality_mask.npy", mmap_mode="r"
    )
    finite = bool(np.isfinite(X).all())
    passed = (X.ndim == 2 and X.shape[1] == 14 and
              X.dtype == np.float32 and len(quality_mask) == len(X) and finite)
    verification.append({
        "Subject": subject_id,
        "Shape": str(X.shape),
        "Dtype": str(X.dtype),
        "Finite": finite,
        "Invalid_64": int((~quality_mask).sum()),
        "Passed": passed,
    })
    del X, quality_mask

df_verify = pd.DataFrame(verification)
display(df_verify)
print("Passed:", int(df_verify["Passed"].sum()), "/", len(df_verify))

,Subject,Shape,Dtype,Finite,Invalid_64,Passed
0,S2,"(389056, 14)",float32,True,0,True
1,S3,"(415552, 14)",float32,True,265,True
2,S4,"(411072, 14)",float32,True,0,True
3,S5,"(400512, 14)",float32,True,0,True
4,S6,"(452544, 14)",float32,True,0,True
5,S7,"(335232, 14)",float32,True,0,True
6,S8,"(349824, 14)",float32,True,0,True
7,S9,"(334272, 14)",float32,True,0,True
8,S10,"(351744, 14)",float32,True,0,True
9,S11,"(334912, 14)",float32,True,0,True


Passed: 15 / 15
